In [ ]:
# Install required libraries if they aren't pre-installed in your environment
# !pip install gradio pandas plotly

import os
import sqlite3
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import drive
from google.colab import ai

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Point directly to your chapter 19 database
db_path = '/content/drive/MyDrive/Colab Notebooks/data/commodity_prices.db'

# Step 2: Dynamically discover the database schema
def read_database_schema():
    if not os.path.exists(db_path):
        print(f"⚠️ Error: Could not find database file at: {db_path}")
        return []

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]

    print("--- 🔍 Live Database Schema Inspection ---")
    print(f"Found tables: {tables}")
    for table in tables:
        df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 1", conn)
        print(f"Columns for '{table}': {df.columns.tolist()}")
    print("------------------------------------------")

    conn.close()
    return tables

tables = read_database_schema()

# Step 3: Data Retrieval & Transformation Rules
def load_data(commodity):
    if not os.path.exists(db_path):
        return None

    conn = sqlite3.connect(db_path)

    # Updated mapping to match discovered table names exactly
    mapping = {
        'WTI crude oil': 'WTI_Crude_Oil',
        'aluminum': 'Aluminum',
        'copper': 'Copper'
    }

    table_to_read = mapping.get(commodity)
    if not table_to_read or table_to_read not in tables:
        # Fallback to search if mapping fails
        table_to_read = next((t for t in tables if commodity.lower().replace(' ', '_') in t.lower()), tables[0])

    df = pd.read_sql_query(f"SELECT * FROM {table_to_read}", conn)
    conn.close()

    # Standardize column names to lowercase
    df.columns = [c.lower() for c in df.columns]

    if 'date' not in df.columns:
        df = df.rename(columns={df.columns[0]: 'date'})

    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')

    # If the table structure is OHLC (which these appear to be based on schema inspection)
    # We ensure 'close' exists as our primary price point
    if 'close' in df.columns:
        df['price'] = df['close']
    else:
        # If it's a generic table, find the first numeric column after date
        df['price'] = df.select_dtypes(include=['number']).iloc[:, 0]

    # Ensure standard OHLC headers exist
    for col in ['open', 'high', 'low', 'close']:
        if col not in df.columns:
            df[col] = df['price']

    return df

def calculate_technical_indicators(df):
    df['ma20'] = df['close'].rolling(window=20).mean()
    df['std20'] = df['close'].rolling(window=20).std()
    df['bb_upper'] = df['ma20'] + (df['std20'] * 2)
    df['bb_lower'] = df['ma20'] - (df['std20'] * 2)
    df['ma50'] = df['close'].rolling(window=50).mean()
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist'] = df['macd'] - df['macd_signal']
    return df

def resample_timeframe(df, view_mode):
    if view_mode == 'Daily':
        return df
    rule = 'W' if view_mode == 'Weekly' else 'ME'
    resampled = df.set_index('date').resample(rule).agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last'
    }).dropna().reset_index()
    return resampled

current_processed_df = None

def analyze_action(commodity, view_mode, active_indicators):
    global current_processed_df
    df = load_data(commodity)
    if df is None or df.empty:
        return go.Figure().update_layout(title_text="Data loading failed."), "Failed to pull data."

    df = resample_timeframe(df, view_mode)
    df = calculate_technical_indicators(df)
    current_processed_df = df

    if 'MACD' in active_indicators:
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            vertical_spacing=0.1, row_heights=[0.7, 0.3])
    else:
        fig = make_subplots(rows=1, cols=1)

    fig.add_trace(go.Candlestick(
        x=df['date'], open=df['open'], high=df['high'],
        low=df['low'], close=df['close'], name='Price'
    ), row=1, col=1)

    if 'Bollinger Bands' in active_indicators:
        fig.add_trace(go.Scatter(x=df['date'], y=df['bb_upper'], line=dict(color='rgba(150,150,150,0.5)', dash='dash'), name='BB Upper'), row=1, col=1)
        fig.add_trace(go.Scatter(x=df['date'], y=df['bb_lower'], line=dict(color='rgba(150,150,150,0.5)', dash='dash'), name='BB Lower'), row=1, col=1)

    if 'Moving Averages' in active_indicators:
        fig.add_trace(go.Scatter(x=df['date'], y=df['ma20'], line=dict(color='orange'), name='20 SMA'), row=1, col=1)
        fig.add_trace(go.Scatter(x=df['date'], y=df['ma50'], line=dict(color='purple'), name='50 SMA'), row=1, col=1)

    if 'MACD' in active_indicators:
        fig.add_trace(go.Scatter(x=df['date'], y=df['macd'], line=dict(color='blue'), name='MACD Line'), row=2, col=1)
        fig.add_trace(go.Scatter(x=df['date'], y=df['macd_signal'], line=dict(color='red'), name='Signal Line'), row=2, col=1)
        fig.add_trace(go.Bar(x=df['date'], y=df['macd_hist'], name='MACD Histogram'), row=2, col=1)

    fig.update_layout(xaxis_rangeslider_visible=False, height=600 if 'MACD' in active_indicators else 400, title_text=f"{commodity} Analysis", template="plotly_white")
    return fig, "Data loaded successfully."

def ai_action(commodity, sensitivity_ratio):
    global current_processed_df
    if current_processed_df is None: return "Analyze data first."
    latest_row = current_processed_df.iloc[-1]
    lookback = min(20, len(current_processed_df))
    past_price = current_processed_df.iloc[-lookback]['close']
    current_price = latest_row['close']
    net_pct = ((current_price - past_price) / past_price) * 100 if past_price != 0 else 0

    prompt = f"Analyze {commodity} at {current_price:.2f}. Trend is {net_pct:.2f}% over last {lookback} periods. Sensitivity: {sensitivity_ratio}% margin drop per 5% rise."
    try:
        return ai.generate_text(prompt=prompt)
    except Exception as e:
        return f"Error: {e}"

with gr.Blocks() as dashboard:
    gr.Markdown("# 📊 Commodity Price Analysis Dashboard")
    with gr.Row():
        commodity_dropdown = gr.Dropdown(choices=["WTI crude oil", "aluminum", "copper"], value="WTI crude oil", label="Target Commodity")
        view_dropdown = gr.Dropdown(choices=["Daily", "Weekly", "Monthly"], value="Daily", label="Chart Resolution")
    with gr.Row():
        indicators_checkbox = gr.CheckboxGroup(choices=["Bollinger Bands", "Moving Averages", "MACD"], value=["Moving Averages"], label="Technical Overlays")
        sensitivity_field = gr.Number(value=1.0, label="Margin reduction (%) per 5% rise")
    analyze_btn = gr.Button("Analyze 📈", variant="primary")
    ai_btn = gr.Button("AI Analysis 🤖", variant="secondary")
    status_field = gr.Textbox(label="Status", interactive=False)
    chart_area = gr.Plot(label="Price Chart")
    ai_area = gr.Markdown(label="AI Report")

    analyze_btn.click(analyze_action, inputs=[commodity_dropdown, view_dropdown, indicators_checkbox], outputs=[chart_area, status_field])
    ai_btn.click(ai_action, inputs=[commodity_dropdown, sensitivity_field], outputs=[ai_area])

dashboard.launch(debug=True)

Mounted at /content/drive
--- 🔍 Live Database Schema Inspection ---
Found tables: ['WTI_Crude_Oil', 'Aluminum', 'Copper']
Columns for 'WTI_Crude_Oil': ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']
Columns for 'Aluminum': ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']
Columns for 'Copper': ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']
------------------------------------------
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea1ad2ca0540a0dffd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the ter